In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os

project_path = '/content/drive/MyDrive/ppe'
print(f"✅ Carpeta encontrada: {os.path.exists(project_path)}")

✅ Carpeta encontrada: True


In [11]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="dSMfDD4uPaMCKEoGOP5q")
project = rf.workspace("cicatriz").project("ppe-factory-bmdcj-alnpk")
version = project.version(1)

# ↓ esto lo guarda en tu carpeta ppe de Drive
dataset = version.download("yolov8", location="/content/drive/MyDrive/ppe/dataset")

print(f"✅ Dataset guardado en Drive: /content/drive/MyDrive/ppe/dataset")

loading Roboflow workspace...
loading Roboflow project...
✅ Dataset guardado en Drive: /content/drive/MyDrive/ppe/dataset


In [12]:
!pip install ultralytics roboflow --quiet

In [13]:
import os

print("Carpetas dentro de dataset:")
for item in os.listdir('/content/drive/MyDrive/ppe/dataset'):
    print(f"  {item}")

Carpetas dentro de dataset:
  roboflow.zip
  README.dataset.txt
  README.roboflow.txt
  test
  train
  data.yaml


In [14]:
import yaml

yaml_path = '/content/drive/MyDrive/ppe/dataset/data.yaml'

data = {
    'train': '/content/drive/MyDrive/ppe/dataset/train/images',
    'val':   '/content/drive/MyDrive/ppe/dataset/test/images',  # ← test como val
    'test':  '/content/drive/MyDrive/ppe/dataset/test/images',
    'nc': 7,
    'names': ['botas.', 'audifonos', 'gafas', 'guantes', 'casco', 'persona', 'chaleco']
}

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("✅ data.yaml corregido")

# Verificar que quedó bien
with open(yaml_path, 'r') as f:
    print(f.read())

✅ data.yaml corregido
names:
- botas.
- audifonos
- gafas
- guantes
- casco
- persona
- chaleco
nc: 7
test: /content/drive/MyDrive/ppe/dataset/test/images
train: /content/drive/MyDrive/ppe/dataset/train/images
val: /content/drive/MyDrive/ppe/dataset/test/images



In [15]:
import os
from ultralytics import YOLO

os.makedirs('/content/drive/MyDrive/ppe/ejecuciones', exist_ok=True)

model = YOLO('yolov8n.pt')
resultado = model.train(
    data='/content/drive/MyDrive/ppe/dataset/data.yaml',
    epochs=5,
    imgsz=640,
    project='/content/drive/MyDrive/ppe/ejecuciones',
    name='yolov8n'
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ppe/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

In [16]:
import os

train_path = '/content/drive/MyDrive/ppe/dataset/train/images'
test_path = '/content/drive/MyDrive/ppe/dataset/test/images'

train_count = len(os.listdir(train_path))
test_count = len(os.listdir(test_path))

print(f"Imágenes en train: {train_count}")
print(f"Imágenes en test:  {test_count}")

Imágenes en train: 9770
Imágenes en test:  286


In [17]:
# Ver métricas completas
import pandas as pd

results_path = '/content/drive/MyDrive/ppe/ejecuciones/yolov8n-4/results.csv'
df = pd.read_csv(results_path)
df.columns = df.columns.str.strip()
print(df[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 
          'metrics/precision(B)', 'metrics/recall(B)']].to_string())

   epoch  metrics/mAP50(B)  metrics/mAP50-95(B)  metrics/precision(B)  metrics/recall(B)
0      1           0.36189              0.23895               0.50902            0.31321
1      2           0.35830              0.23029               0.57088            0.30774
2      3           0.38683              0.24322               0.50564            0.35723
3      4           0.34640              0.23579               0.62793            0.31467
4      5           0.37107              0.25811               0.62769            0.32884


In [ ]:
# ============================================================
# CELDA: Correr Streamlit con LocalTunnel (sin cuenta)
# ============================================================
!pip install streamlit -q
!npm install -g localtunnel -q

# Corre Streamlit en background
import subprocess
subprocess.Popen([
    "streamlit", "run", "/content/app.py",
    "--server.port", "8501",
    "--server.headless", "true"
])

import time
time.sleep(3)

# Abre el túnel
!lt --port 8501 &

# Muestra la IP del servidor (necesaria para autenticar localtunnel)
import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"\n✅ Cuando abras el link de localtunnel, ingresa esta IP: {ip}")

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 5s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏npm notice
npm notice New major version of npm available! 10.8.2 -> 11.13.0
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.13.0
npm notice To update run: npm install -g npm@11.13.0
npm notice
⠏your url is: https://social-chefs-start.loca.lt
